---
## Task 3 — Shots on Target: Group Stage vs. Knockout Stage

### Analytic question formulation
Do teams register a significantly different number of **shots on target per match** in
**knockout-stage** matches compared to **group-stage** matches (e.g. because knockout games are
more cagey/high-stakes)?

### Data wrangling
Fetch the `TeamMatch` sheet from the clean dataset and keep the `Stage` and `SOT`
(shots on target) columns. The **population** is every team-match record with a non-missing SOT
value.


In [1]:
import pandas as pd
import numpy as np
import re
from scipy import stats
CLEAN_PATH = "../World_Cup_2026_clean.xlsx"
team_match_t2 = pd.read_excel(CLEAN_PATH, sheet_name="TeamMatch")
pop_sot = team_match_t2.dropna(subset=["SOT"])[["Stage", "SOT"]]

print("Population size (team-match records):", len(pop_sot))
print(pop_sot.groupby("Stage")["SOT"].mean())

Population size (team-match records): 206
Stage
Group Stage    4.152778
Knockout       4.306452
Name: SOT, dtype: float64


### Data preparation and sampling
**Population:** all 206 team-match SOT records (144 group-stage, 62 knockout).
**Sample:** a **stratified random sample** — 25 records drawn independently at random from each
stage — so both groups are guaranteed adequate, balanced representation for the two-sample test.


In [2]:
grp_sample = pop_sot[pop_sot.Stage == "Group Stage"].sample(n=25, random_state=1)["SOT"]
ko_sample  = pop_sot[pop_sot.Stage == "Knockout"].sample(n=25, random_state=1)["SOT"]
print("Group-stage sample n:", len(grp_sample), " Knockout sample n:", len(ko_sample))

Group-stage sample n: 25  Knockout sample n: 25


### Descriptive statistics

In [3]:
for name, s in [("Group Stage", grp_sample), ("Knockout", ko_sample)]:
    print(f"\n{name}:")
    print(s.describe())


Group Stage:
count    25.000000
mean      3.400000
std       2.101587
min       0.000000
25%       2.000000
50%       3.000000
75%       5.000000
max       8.000000
Name: SOT, dtype: float64

Knockout:
count    25.000000
mean      4.240000
std       2.504662
min       0.000000
25%       2.000000
50%       4.000000
75%       6.000000
max      10.000000
Name: SOT, dtype: float64


### Inferential statistics — Confidence intervals (95%, each stage)

In [4]:
for name, s in [("Group Stage", grp_sample), ("Knockout", ko_sample)]:
    m, sem = s.mean(), stats.sem(s)
    ci = stats.t.interval(0.95, df=len(s) - 1, loc=m, scale=sem)
    print(f"{name}: mean = {m:.3f}, 95% CI = ({ci[0]:.3f}, {ci[1]:.3f})")

Group Stage: mean = 3.400, 95% CI = (2.533, 4.267)
Knockout: mean = 4.240, 95% CI = (3.206, 5.274)


### Inferential statistics — Two-sample t-Test (Welch's, unequal variances)
H₀: μ(group) = μ(knockout)  vs.  H₁: μ(group) ≠ μ(knockout)


In [5]:
t_stat, p_val = stats.ttest_ind(grp_sample, ko_sample, equal_var=False)
print(f"t-statistic = {t_stat:.3f}, p-value = {p_val:.4f}")
alpha = 0.05
print("Conclusion:", "Reject H0" if p_val < alpha else "Fail to reject H0",
      "at the 5% significance level.")

t-statistic = -1.285, p-value = 0.2053
Conclusion: Fail to reject H0 at the 5% significance level.


**Interpretation:** Knockout-stage teams averaged more shots on target in this sample
(4.24 vs 3.40), but p = 0.205 (> 0.05) means the difference is not statistically significant —
it's plausibly just sampling variation. The two CIs also overlap substantially.
